# PCIe 读写一致性测试

这份 notebook 通过 XDMA 的 memory mapped PCIe 通道测试 FPGA BRAM 的读写一致性：

- 使用 `h2c_0` 将测试数据通过 PCIe 写入 FPGA。
- 使用 `c2h_0` 从相同 PCIe/AXI 地址读回数据。
- 逐字节比较写入文件和读回文件，失败时打印最早的不一致位置。

当前工程 `xdma_bram` 的 BRAM 地址段在 BD 中映射到 `0x10000000`，范围是 `32KB`，所以默认测试地址都在 `[0x10000000, 0x10008000)` 内。

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import time
from dataclasses import dataclass

# 如果 Jupyter 的工作目录就是本 notebook 所在目录，直接使用当前目录。
# 如果从仓库根目录或其他目录启动，可手动改 NOTEBOOK_DIR。
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "bin" / "xdma_rw.exe").exists():
    repo_candidate = Path("/home/triton/task/YOLO_on_FPGA/fpga/local/tests/xdma_bram")
    if (repo_candidate / "bin" / "xdma_rw.exe").exists():
        NOTEBOOK_DIR = repo_candidate

XDMA_EXE = NOTEBOOK_DIR / "bin" / "xdma_rw.exe"
DATA_DIR = NOTEBOOK_DIR / "data"
READBACK_DIR = NOTEBOOK_DIR / "readbacks"
DATA_DIR.mkdir(exist_ok=True)
READBACK_DIR.mkdir(exist_ok=True)

# 来自 fpga/local/scripts/xdma_bram/1_bd.tcl 和生成的 xdma_bram.bd。
BRAM_BASE = 0x10000000
BRAM_SIZE = 0x8000
CHANNEL = 0
CMD_TIMEOUT_SEC = 30

assert XDMA_EXE.exists(), f"找不到 XDMA 工具: {XDMA_EXE}"
print(f"XDMA 工具: {XDMA_EXE}")
print(f"测试目录: {NOTEBOOK_DIR}")
print(f"BRAM PCIe 地址范围: 0x{BRAM_BASE:08X}..0x{BRAM_BASE + BRAM_SIZE:08X}")

In [ ]:
@dataclass
class CmdResult:
    cmd: list[str]
    returncode: int
    stdout: str
    stderr: str
    elapsed_sec: float

    @property
    def ok(self) -> bool:
        return self.returncode == 0


def run_cmd(cmd: list[str], timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    start = time.perf_counter()
    cp = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )
    return CmdResult(
        cmd=cmd,
        returncode=cp.returncode,
        stdout=cp.stdout,
        stderr=cp.stderr,
        elapsed_sec=time.perf_counter() - start,
    )


def print_result(name: str, result: CmdResult) -> None:
    status = "PASS" if result.ok else "FAIL"
    print(f"{name}: {status}, ret={result.returncode}, elapsed={result.elapsed_sec:.3f}s")
    print("CMD:", " ".join(result.cmd))
    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())


def make_pattern(size: int, pattern: str, seed: int = 0) -> bytes:
    if pattern == "zeros":
        return bytes(size)
    if pattern == "ones":
        return bytes([0xFF]) * size
    if pattern == "counter":
        return bytes((seed + i) & 0xFF for i in range(size))
    if pattern == "inverse":
        return bytes((~(seed + i)) & 0xFF for i in range(size))
    if pattern == "walking":
        return bytes(1 << ((seed + i) & 7) for i in range(size))
    if pattern == "lcg":
        value = seed & 0xFFFFFFFF
        out = bytearray()
        for _ in range(size):
            value = (1664525 * value + 1013904223) & 0xFFFFFFFF
            out.append((value >> 24) & 0xFF)
        return bytes(out)
    raise ValueError(f"未知 pattern: {pattern}")


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def validate_range(offset: int, size: int) -> None:
    assert offset >= 0, "offset 不能为负数"
    assert size > 0, "size 必须大于 0"
    assert offset % 4 == 0, f"offset 建议 4 字节对齐: 0x{offset:X}"
    assert size % 4 == 0, f"size 建议 4 字节对齐: 0x{size:X}"
    assert offset + size <= BRAM_SIZE, (
        f"测试范围越过 BRAM: offset=0x{offset:X}, size=0x{size:X}, BRAM_SIZE=0x{BRAM_SIZE:X}"
    )


def xdma_write(channel: int, addr: int, payload_path: Path) -> CmdResult:
    return run_cmd([
        str(XDMA_EXE),
        f"h2c_{channel}",
        "write",
        hex(addr),
        "-b",
        "-f",
        str(payload_path),
    ])


def xdma_read(channel: int, addr: int, size: int, readback_path: Path) -> CmdResult:
    if readback_path.exists():
        readback_path.unlink()
    return run_cmd([
        str(XDMA_EXE),
        f"c2h_{channel}",
        "read",
        hex(addr),
        "-l",
        hex(size),
        "-b",
        "-f",
        str(readback_path),
    ])


def compare_bytes(expected: bytes, actual_path: Path, max_diff: int = 16) -> tuple[bool, str]:
    if not actual_path.exists():
        return False, f"未生成读回文件: {actual_path}"

    actual = actual_path.read_bytes()
    if expected == actual:
        return True, f"一致: {len(expected)} bytes, sha256={sha256_file(actual_path)}"

    lines = [f"不一致: write_len={len(expected)}, read_len={len(actual)}"]
    for index, (write_byte, read_byte) in enumerate(zip(expected, actual)):
        if write_byte != read_byte:
            lines.append(f"0x{index:08X}: write=0x{write_byte:02X}, read=0x{read_byte:02X}")
            if len(lines) > max_diff:
                break
    if len(expected) != len(actual):
        lines.append("读回长度与写入长度不同")
    return False, "\n".join(lines)


def run_pcie_rw_case(
    offset: int,
    size: int,
    pattern: str = "counter",
    seed: int = 0,
    channel: int = CHANNEL,
    settle_sec: float = 0.02,
) -> dict:
    validate_range(offset, size)
    addr = BRAM_BASE + offset
    payload = make_pattern(size=size, pattern=pattern, seed=seed)
    case_name = f"ch{channel}_off{offset:05X}_size{size:05X}_{pattern}_seed{seed:08X}"
    write_path = DATA_DIR / f"write_{case_name}.bin"
    readback_path = READBACK_DIR / f"read_{case_name}.bin"
    write_path.write_bytes(payload)

    print(f"\n=== PCIe RW case: addr=0x{addr:08X}, offset=0x{offset:X}, size={size}, pattern={pattern} ===")
    write_result = xdma_write(channel, addr, write_path)
    print_result("WRITE h2c", write_result)
    if settle_sec > 0:
        time.sleep(settle_sec)

    read_result = xdma_read(channel, addr, size, readback_path)
    print_result("READ  c2h", read_result)

    match, detail = compare_bytes(payload, readback_path)
    passed = write_result.ok and read_result.ok and match
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)

    return {
        "pass": passed,
        "offset": offset,
        "addr": addr,
        "size": size,
        "pattern": pattern,
        "seed": seed,
        "channel": channel,
        "write_ok": write_result.ok,
        "read_ok": read_result.ok,
        "match": match,
        "write_file": write_path,
        "readback_file": readback_path,
        "detail": detail,
        "write_result": write_result,
        "read_result": read_result,
    }


def summarize(results: list[dict]) -> None:
    print("\n" + "=" * 88)
    print("PCIe RW consistency summary")
    print("=" * 88)
    for i, result in enumerate(results):
        status = "PASS" if result["pass"] else "FAIL"
        print(
            f"{i:02d} {status} "
            f"addr=0x{result['addr']:08X} size=0x{result['size']:X} "
            f"pattern={result['pattern']} write_ok={result['write_ok']} "
            f"read_ok={result['read_ok']} match={result['match']}"
        )
    passed = sum(1 for result in results if result["pass"])
    print(f"\nPASS {passed}/{len(results)}")

In [ ]:
# 最小冒烟测试：写 4 字节，再从同一 PCIe/AXI 地址读回 4 字节。
single_result = run_pcie_rw_case(offset=0x100, size=4, pattern="counter", seed=0x12)
assert single_result["pass"], single_result["detail"]
single_result

In [ ]:
# 批量一致性测试：覆盖不同地址、长度和数据 pattern。
TEST_CASES = [
    {"offset": 0x0000, "size": 0x0004, "pattern": "counter", "seed": 0x11},
    {"offset": 0x0100, "size": 0x0040, "pattern": "counter", "seed": 0x20},
    {"offset": 0x0200, "size": 0x0100, "pattern": "walking", "seed": 0x00},
    {"offset": 0x0400, "size": 0x0400, "pattern": "lcg", "seed": 0x1234},
    {"offset": 0x1000, "size": 0x1000, "pattern": "inverse", "seed": 0xA5},
    {"offset": 0x7000, "size": 0x0800, "pattern": "lcg", "seed": 0x55AA},
]

batch_results = [run_pcie_rw_case(**case) for case in TEST_CASES]
summarize(batch_results)
assert all(result["pass"] for result in batch_results), "存在 PCIe 读写不一致的测试用例"
batch_results

In [ ]:
# 覆写一致性测试：同一地址连续写入不同 pattern，确认读回不是旧数据。
overwrite_results = []
overwrite_results.append(run_pcie_rw_case(offset=0x1800, size=0x200, pattern="zeros", seed=0))
overwrite_results.append(run_pcie_rw_case(offset=0x1800, size=0x200, pattern="ones", seed=0))
overwrite_results.append(run_pcie_rw_case(offset=0x1800, size=0x200, pattern="lcg", seed=0xDEADBEEF))

summarize(overwrite_results)
assert all(result["pass"] for result in overwrite_results), "覆写测试失败"
overwrite_results

In [ ]:
# 可选压力测试：需要时手动执行。
# 每轮使用 4 字节对齐的伪随机 offset/size，仍限制在 32KB BRAM 地址段内。
def run_stress(rounds: int = 16, max_size: int = 0x1000) -> list[dict]:
    results = []
    value = 0xC001D00D
    for round_id in range(rounds):
        value = (1103515245 * value + 12345) & 0x7FFFFFFF
        size = ((value % max_size) & ~0x3) or 4
        value = (1103515245 * value + 12345) & 0x7FFFFFFF
        max_offset = BRAM_SIZE - size
        offset = (value % max_offset) & ~0x3
        results.append(run_pcie_rw_case(offset=offset, size=size, pattern="lcg", seed=round_id))
    summarize(results)
    assert all(result["pass"] for result in results), "压力测试存在失败用例"
    return results

# stress_results = run_stress(rounds=32, max_size=0x1000)